# Baseline 코드

## 1. 불러오기

In [3]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

SEED = 42
DATA_DIR = "./bank/"                # ★ 꼭 파일의 경로를 설정하세요.
TARGET, ID = "Exited", "id"

train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")
sub   = pd.read_csv("sample_submission.csv")
print("train", train.shape, "| test", test.shape)
train.head(3)

train (35000, 13) | test (15000, 12)


,id,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,Lin,693.0,Germany,Male,59.0,10,126446.75,3,0.0,0.0,8153.79,1
1,2,Frolov,790.0,Spain,Male,34.0,9,0.00,2,0.0,0.0,121389.72,0
2,3,Nwankwo,649.0,France,Female,31.0,9,0.00,2,1.0,1.0,73454.56,0


## 2. 간단 EDA

In [4]:
print(f"양성(해지) 비율 = {train[TARGET].mean()*100:.2f}%")
print("무작위 예측 AUC = 0.5000\n")

na = train.isna().mean()[lambda s: s > 0]
print("결측 있는 컬럼(%)\n" + ((na*100).round(2).to_string() if len(na) else "없음"), "\n")

cat_all = [c for c in train.columns
           if c not in (ID, TARGET) and not pd.api.types.is_numeric_dtype(train[c])]
print("범주형 컬럼과 고유값 수:", {c: train[c].nunique() for c in cat_all})

양성(해지) 비율 = 21.16%
무작위 예측 AUC = 0.5000

결측 있는 컬럼(%)
CreditScore         5.08
EstimatedSalary    12.31 

범주형 컬럼과 고유값 수: {'Surname': 2202, 'Geography': 3, 'Gender': 2}


## 3. 전처리

In [5]:
num_cols = [c for c in train.columns
            if c not in (ID, TARGET) and pd.api.types.is_numeric_dtype(train[c])]

# Surname은 고유값이 2천 개가 넘어 일단 버린다.
cat_cols = [c for c in cat_all if train[c].nunique() <= 20]
dropped  = [c for c in cat_all if c not in cat_cols]

print(f"수치형 {len(num_cols)}개  : {num_cols}")
print(f"범주형 {len(cat_cols)}개  : {cat_cols}")
print(f"버린 컬럼           : {dropped}")

pre = ColumnTransformer([
    # 결측 -> 중앙값 대치,  스케일 차이 -> 표준화 (선형 모델은 둘 다 필수)
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc",  StandardScaler())]), num_cols),
    # 문자열 -> 원-핫 인코딩.  handle_unknown="ignore" 가 있어야 test에만 있는 값 처리
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh",  OneHotEncoder(handle_unknown="ignore",
                                            sparse_output=False))]), cat_cols),
], remainder="drop")

def make_model():
    return Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000, random_state=SEED))])

X, Xte = train.drop(columns=[ID, TARGET]), test.drop(columns=[ID])
y = train[TARGET].to_numpy()
print(f"\n전처리 후 피처 수: {make_model().fit(X, y).named_steps['pre'].transform(X).shape[1]}")

수치형 8개  : ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']
범주형 2개  : ['Geography', 'Gender']
버린 컬럼           : ['Surname']

전처리 후 피처 수: 13


## 4. 학습

In [6]:
skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
oof, fold_auc = np.zeros(len(X)), []
for k, (tr_i, va_i) in enumerate(skf.split(X, y), 1):
    m = make_model().fit(X.iloc[tr_i], y[tr_i])
    oof[va_i] = m.predict_proba(X.iloc[va_i])[:, 1]      # ★ [:,1] = 양성 확률
    fold_auc.append(roc_auc_score(y[va_i], oof[va_i]))
    print(f"  fold {k}  AUC={fold_auc[-1]:.5f}")

print(f"\nOOF AUC  = {roc_auc_score(y, oof):.5f}")

  fold 1  AUC=0.80570
  fold 2  AUC=0.82762
  fold 3  AUC=0.82112
  fold 4  AUC=0.82726
  fold 5  AUC=0.81763

OOF AUC  = 0.81981


## 5. 제출

In [7]:
model = make_model().fit(X, y)
proba = model.predict_proba(Xte)[:, 1]        # ★ predict()가 아니라 predict_proba()로 확률 예측

submission = pd.DataFrame({ID: test[ID], TARGET: proba})

submission.to_csv("submission.csv", index=False)
print("submission.csv 생성 완료")

submission.csv 생성 완료
